In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
!pip install gender-guesser openai pandas pyarrow -q
# import gender_enrichment

In [ ]:
#!/usr/bin/env python3
"""
gender_enrichment.py  —  Google Colab   (full-name + country context, dual-LLM)
===============================================================================
Rewrite: gender is inferred by dual-LLM CONSENSUS from the FULL author name
plus the author's COUNTRY as context, uniformly for EVERY name (the offline
gender-guesser is kept only as an optional cross-check column, not the primary).

WHY THE REWRITE
  The prior cache keyed on name FRAGMENTS ("Fang", "Boniface"), so the LLM judged
  bare given names with no surname signal -> low consensus. Sending the whole
  name + country lets the models actually agree. One method, applied to all names,
  is also far more defensible than a gender-guesser/LLM patchwork.

KEY = (full_name, country)  because "Fang Wei" in CN vs elsewhere differ. Country
is a HINT for disambiguation only; you still MEASURE gender and still validate
precision per bloc (country context can inflate coverage via stereotype — the
validation harness is how you catch that).

Gender here is name-typical presentation, NOT identity. Keep that in the paper.

SETUP
  !pip install openai gender-guesser pandas pyarrow -q
  # Colab secrets: OPENAI_API_KEY, DEEPSEEK_API_KEY
  # Use a FRESH cache path (the old fragment cache is not reused).
"""

import json, re, sys, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import pandas as pd

# =========================================================================== #
# CONFIG                                                                        #
# =========================================================================== #

IN_PARQUET   = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/ror_enrich.parquet"
IN_PARQUET2   = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/with_venue.parquet"
OUT_PARQUET  = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/with_gender.parquet"
NAME_CACHE   = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/gender_name_cache/gender_fullname_cache.jsonl"
VALID_CSV    = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/gender_name_cache/gender_validation.csv"

# IN_PARQUET  = "/content/drive/MyDrive/.../with_doi.parquet"    # a parquet that HAS study_bloc + au_country_vec
# OUT_PARQUET = "/content/drive/MyDrive/.../with_gender.parquet"
# NAME_CACHE  = "/content/drive/MyDrive/.../gender_fullname_cache.jsonl"   # FRESH path
# VALID_CSV   = "/content/drive/MyDrive/.../gender_validation.csv"
Path(NAME_CACHE).parent.mkdir(parents=True, exist_ok=True)

PROB_MIN    = 0.90
LLM_BATCH   = 30             # names per LLM call (full name + country -> keep modest)
LLM_WORKERS = 6
ADD_OFFLINE_XCHECK = True    # also store gender-guesser opinion as a separate column

MAGHREB = {"MA","DZ","TN","LY","EG","MR"}
SSA = {"AO","BJ","BW","BF","BI","CV","CM","CF","TD","KM","CG","CD","CI","GQ","ER",
       "SZ","ET","GA","GM","GH","GN","GW","KE","LS","LR","MG","MW","ML","MZ","NA",
       "NE","NG","RW","ST","SN","SC","SL","SO","ZA","SS","SD","TZ","TG","UG","ZM","ZW"}

try:
    from google.colab import userdata
    _secret = userdata.get
except Exception:
    import os
    _secret = os.environ.get


def S(x):
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)

def ckey(full, country):
    """Cache key: full name + country hint, so identical names in different
    countries are resolved (and cached) separately."""
    return f"{S(full).strip()}||{S(country).strip().upper()}"


# =========================================================================== #
# 1. collect (full_name, country) pairs, position-aligned to au_country_vec    #
# =========================================================================== #
def author_country_list(row):
    """Per-author country (first listed) aligned to oa_authors order, using the
    filled vector if present."""
    col = ("au_country_vec_filled" if "au_country_vec_filled" in row.index
           and S(row.get("au_country_vec_filled")) else "au_country_vec")
    vec = S(row.get(col)).split("|")   # '|' between authors, ';' within
    out = []
    for slot in vec:
        parts = [c for c in slot.split(";") if c]
        out.append(parts[0] if parts else "")
    return out

def all_name_country(df):
    """Distinct (full_name, country) pairs across the corpus."""
    pairs = {}
    for _, r in df.iterrows():
        authors = [a.strip() for a in S(r.get("oa_authors")).split("|")]
        countries = author_country_list(r)
        for i, full in enumerate(authors):
            if not full:
                continue
            country = countries[i] if i < len(countries) else ""
            pairs.setdefault(ckey(full, country), (full, country))
    print(f"[gender] {len(pairs):,} distinct (name, country) pairs", file=sys.stderr)
    return pairs


def load_cache():
    done = {}
    if Path(NAME_CACHE).exists():
        for line in open(NAME_CACHE):
            try:
                d = json.loads(line); done[d["key"]] = d
            except Exception:
                pass
    print(f"[gender] {len(done):,} pairs cached", file=sys.stderr)
    return done

def _append(fh, rec):
    fh.write(json.dumps(rec, ensure_ascii=False) + "\n"); fh.flush()


# =========================================================================== #
# 2. dual-LLM consensus on FULL name + country, for ALL pairs                 #
# =========================================================================== #
def _make_client(which):
    from openai import OpenAI
    if which == "deepseek":
        return OpenAI(api_key=_secret("DEEPSEEK_API_KEY"),
                      base_url="https://api.deepseek.com"), "deepseek-chat"
    return OpenAI(api_key=_secret("OPENAI_API_KEY")), "gpt-4o-mini"

SYS = ("You infer the likely gender of a researcher from their FULL name, using "
       "the given name AND family-name origin/language. A country of affiliation is "
       "given as a HINT to disambiguate romanizations; do NOT assume gender from "
       "country alone. Return ONLY a JSON array, one object per input in the SAME "
       'order: [{"gender":"male|female|unknown","prob":0.0-1.0}]. Use "unknown" for '
       "initials-only, unisex, organizational, or names you cannot judge. Do NOT "
       "guess to fill — reflect genuine low confidence in prob.")

def _llm_batch(client, model, batch):
    lines = [f'{i+1}. name="{full}" country={country or "?"}'
             for i, (full, country) in enumerate(batch)]
    user = "Infer gender for each:\n" + "\n".join(lines)
    for attempt in range(4):
        try:
            r = client.chat.completions.create(
                model=model, temperature=0, max_tokens=3500,
                messages=[{"role": "system", "content": SYS},
                          {"role": "user", "content": user}])
            txt = re.sub(r"^```(json)?|```$", "", r.choices[0].message.content.strip(),
                         flags=re.MULTILINE).strip()
            arr = json.loads(txt)
            if len(arr) == len(batch):
                out = []
                for o in arr:
                    g = o.get("gender", "unknown")
                    out.append((g if g in ("male", "female") else "unknown",
                                float(o.get("prob") or 0)))
                return out
        except Exception:
            time.sleep(2 ** attempt)
    return [("unknown", 0.0)] * len(batch)

def run_consensus(pairs, cache):
    oa_client, oa_model = _make_client("openai")
    ds_client, ds_model = _make_client("deepseek")
    todo_keys = [k for k in pairs if k not in cache]
    print(f"[llm] {len(todo_keys):,} pairs -> OpenAI + DeepSeek (full name + country)",
          file=sys.stderr)
    fh = open(NAME_CACHE, "a")

    items = [(k, pairs[k][0], pairs[k][1]) for k in todo_keys]
    batches = [items[i:i + LLM_BATCH] for i in range(0, len(items), LLM_BATCH)]

    def process(batch):
        payload = [(full, country) for _, full, country in batch]
        oa = _llm_batch(oa_client, oa_model, payload)
        ds = _llm_batch(ds_client, ds_model, payload)
        recs = []
        for (k, full, country), (g1, p1), (g2, p2) in zip(batch, oa, ds):
            agree = (g1 == g2) and g1 in ("male", "female") and p1 >= PROB_MIN and p2 >= PROB_MIN
            base = {"key": k, "name": full, "country": country,
                    "openai": f"{g1}:{p1}", "deepseek": f"{g2}:{p2}"}
            if agree:
                base.update({"provider": "llm_consensus", "gender": g1,
                             "prob": round(min(p1, p2), 3)})
            else:
                base.update({"provider": "llm_nocons", "gender": "unknown", "prob": 0.0})
            recs.append(base)
        return recs

    done = 0
    with ThreadPoolExecutor(max_workers=LLM_WORKERS) as ex:
        futs = [ex.submit(process, b) for b in batches]
        for fut in as_completed(futs):
            for rec in fut.result():
                _append(fh, rec); cache[rec["key"]] = rec
            done += 1
            if done % 20 == 0:
                print(f"[llm] {done:,}/{len(batches):,} batches", file=sys.stderr)
    fh.close()
    return cache


# =========================================================================== #
# 3. optional offline cross-check (stored separately, NOT the primary)         #
# =========================================================================== #
def add_offline_xcheck(pairs, cache):
    import gender_guesser.detector as gg
    det = gg.Detector(case_sensitive=False)
    conf = {"male": 0.95, "female": 0.95, "mostly_male": 0.75,
            "mostly_female": 0.75, "andy": 0.5, "unknown": 0.0}
    for k, (full, _) in pairs.items():
        d = cache.get(k)
        if not d or "offline_gender" in d:
            continue
        toks = [t for t in re.split(r"[ \-]", full) if len(t) > 1]
        gg_g, gg_p = "unknown", 0.0
        for t in toks:                       # try each token, take first confident
            g = det.get_gender(t.strip(".,"))
            base = "female" if "female" in g else ("male" if "male" in g else "unknown")
            if base != "unknown":
                gg_g, gg_p = base, conf.get(g, 0.0); break
        d["offline_gender"] = gg_g; d["offline_prob"] = gg_p
    return cache


# =========================================================================== #
# 4. attach to papers (position-aligned; primary = LLM consensus)              #
# =========================================================================== #
def gated(full, country, cache):
    d = cache.get(ckey(full, country))
    if not d or d.get("gender") not in ("male", "female") or d.get("prob", 0) < PROB_MIN:
        return "unknown"
    return "M" if d["gender"] == "male" else "F"

def attach(df, cache):
    def row(r):
        authors = [a.strip() for a in S(r.get("oa_authors")).split("|")]
        countries = author_country_list(r)
        pos = S(r.get("au_position_vec")).split("|")
        first_g = last_g = "unknown"; genders = []
        for i, full in enumerate(authors):
            if not full:
                continue
            country = countries[i] if i < len(countries) else ""
            g = gated(full, country, cache); genders.append(g)
            p = pos[i] if i < len(pos) else ""
            if p == "first": first_g = g
            if p == "last":  last_g = g
        known = [g for g in genders if g in ("M", "F")]
        return pd.Series({
            "first_author_gender": first_g,
            "last_author_gender": last_g,
            "share_women_known": (known.count("F") / len(known)) if known else None,
            "n_authors_gender_known": len(known),
            "n_authors_total": len([a for a in authors if a]),
        })
    out = pd.concat([df, df.apply(row, axis=1)], axis=1)
    out.to_parquet(OUT_PARQUET, index=False)
    print(f"[write] {OUT_PARQUET}", file=sys.stderr)
    resolved = sum(1 for d in cache.values()
                   if d.get("gender") in ("male", "female") and d.get("prob", 0) >= PROB_MIN)
    print(f"[coverage] pairs resolved to M/F: {resolved:,}/{len(cache):,} "
          f"({resolved/max(len(cache),1):.1%})", file=sys.stderr)
    # agreement rate between LLM consensus and offline cross-check (a QC signal)
    if ADD_OFFLINE_XCHECK:
        both = [(d["gender"], d.get("offline_gender")) for d in cache.values()
                if d.get("gender") in ("male", "female")
                and d.get("offline_gender") in ("male", "female")]
        if both:
            agree = sum(1 for a, b in both if a == b) / len(both)
            print(f"[qc] LLM vs offline agreement where both resolve: {agree:.1%} "
                  f"(n={len(both):,})", file=sys.stderr)
    return out


# =========================================================================== #
# 5. VALIDATION HARNESS — precision + coverage per bloc, go/no-go gate         #
# =========================================================================== #
def dominant_bloc(sb):
    parts = {x for x in S(sb).split(";") if x}
    for b in ("ssa", "maghreb"):
        if b in parts: return b
    return "other"

def make_validation(out, n_per=80, seed=42):
    if "study_bloc" not in out.columns:
        print("[validate] study_bloc absent — run on a parquet with study-location.",
              file=sys.stderr)
        return
    d = out.copy(); d["_bloc"] = d["study_bloc"].map(dominant_bloc)
    parts = [g.sample(min(n_per, len(g)), random_state=seed) for _, g in d.groupby("_bloc")]
    s = pd.concat(parts).copy()
    s["first_author_name"] = s["oa_authors"].map(lambda v: S(v).split("|")[0])
    s = s[["oa_id", "first_author_name", "first_author_gender", "_bloc"]].copy()
    s["gold_gender"] = ""
    s.to_csv(VALID_CSV, index=False)
    print(f"[validate] wrote {len(s):,} rows -> {VALID_CSV} (fill gold_gender: M/F/U)",
          file=sys.stderr)

def score_validation(path=VALID_CSV):
    v = pd.read_csv(path, dtype=str).fillna("")
    print("\n[validation] per bloc (precision among M/F; coverage = share guessed):")
    print(f"{'bloc':<9}{'n':>6}{'coverage':>10}{'precision':>11}")
    gate = {}
    for bloc, g in v.groupby("_bloc"):
        guessed = g[g["first_author_gender"].isin(["M", "F"])]
        coverage = len(guessed) / len(g) if len(g) else 0
        gold_ok = guessed[guessed["gold_gender"].isin(["M", "F"])]
        correct = (gold_ok["first_author_gender"] == gold_ok["gold_gender"]).sum()
        precision = correct / len(gold_ok) if len(gold_ok) else float("nan")
        gate[bloc] = coverage
        print(f"{bloc:<9}{len(g):>6}{coverage:>10.1%}{precision:>11.3f}")
    if "ssa" in gate and "other" in gate:
        gap = abs(gate.get("ssa", 0) - gate.get("other", 0))
        verdict = ("OK to compare blocs" if gap < 0.15 else
                   "DO NOT compare blocs — coverage gap too large; report global/North only")
        print(f"\n[gate] SSA vs other coverage gap = {gap:.1%}  ->  {verdict}")


# =========================================================================== #
if __name__ == "__main__":
    df1 = pd.read_parquet(IN_PARQUET)
    df2 = pd.read_parquet(IN_PARQUET2)[['oa_id','oa_authors']]
    df = pd.merge(df1, df2, on='oa_id', how='left')

    print(f"[in] {len(df):,} rows", file=sys.stderr)
    pairs = all_name_country(df)
    cache = load_cache()
    cache = run_consensus(pairs, cache)
    if ADD_OFFLINE_XCHECK:
        cache = add_offline_xcheck(pairs, cache)
    out = attach(df, cache)
    make_validation(out)
    # after a human fills gold_gender in VALID_CSV:  score_validation()

In [ ]:
pairs

In [ ]:
cache

In [ ]:
out

In [ ]:
# #!/usr/bin/env python3
# """
# gender_enrichment.py  —  Google Colab   (dual-LLM consensus edition)
# ====================================================================
# Name -> gender, done defensibly, with OpenAI + DeepSeek run TOGETHER for
# consensus. Inference is biased against African / East Asian / initials-only
# names — the researchers this paper is about — so the design is:
#   * infer only above a confidence gate; everything else is UNKNOWN (never impute)
#   * run BOTH LLMs; accept a gender only where they AGREE (consensus)
#   * validate per region and gate cross-bloc comparison on coverage parity.

# LAYERS (all cached per unique name, resumable):
#   1. offline dictionary (gender-guesser) — free first pass, Europe-heavy
#   2. dual-LLM consensus (OpenAI gpt-4o-mini + DeepSeek) on names still unresolved
#      — batched, full name used, results agreed across both models

# Gender here is name-typical presentation, NOT identity. Keep that in the paper.

# REUSABLE: the per-name cache + per-author columns support later studies of
# gender disparities, collaboration, and women's integration across scenarios.

# SETUP
#   !pip install gender-guesser openai pandas pyarrow -q
#   # Colab secrets: OPENAI_API_KEY, DEEPSEEK_API_KEY
# """

# import json, re, sys, time
# from concurrent.futures import ThreadPoolExecutor, as_completed
# from pathlib import Path
# import pandas as pd

# # =========================================================================== #
# # CONFIG                                                                        #
# # =========================================================================== #
# IN_PARQUET   = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/llm_screen/screened_llm.parquet"
# OUT_PARQUET  = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/with_gender.parquet"
# NAME_CACHE   = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/gender_name_cache/gender_name_cache.jsonl"
# VALID_CSV    = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/gender_name_cache/gender_validation.csv"

# Path(NAME_CACHE).parent.mkdir(parents=True, exist_ok=True)
# Path(VALID_CSV).parent.mkdir(parents=True, exist_ok=True)

# # IN_PARQUET  = "/content/drive/MyDrive/.../with_doi.parquet"   # use a parquet that HAS study_bloc
# # OUT_PARQUET = "/content/drive/MyDrive/.../with_gender.parquet"
# # NAME_CACHE  = "/content/drive/MyDrive/.../gender_name_cache.jsonl"
# # VALID_CSV   = "/content/drive/MyDrive/.../gender_validation.csv"

# PROB_MIN    = 0.90           # below this -> unknown (do NOT guess)
# RUN_OFFLINE = True           # gender-guesser first pass
# RUN_LLM     = True           # dual-LLM consensus on the residual
# LLM_BATCH   = 40             # names per LLM call
# LLM_WORKERS = 6              # concurrent batches

# MAGHREB = {"MA","DZ","TN","LY","EG","MR"}
# SSA = {"AO","BJ","BW","BF","BI","CV","CM","CF","TD","KM","CG","CD","CI","GQ","ER",
#        "SZ","ET","GA","GM","GH","GN","GW","KE","LS","LR","MG","MW","ML","MZ","NA",
#        "NE","NG","RW","ST","SN","SC","SL","SO","ZA","SS","SD","TZ","TG","UG","ZM","ZW"}

# try:
#     from google.colab import userdata
#     _secret = userdata.get
# except Exception:
#     import os
#     _secret = os.environ.get


# def S(x):
#     return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)

# def first_name(full):
#     """Given name for the offline dict. '' for initials-only."""
#     full = S(full).strip()
#     if not full:
#         return ""
#     tok = full.split()[-1].strip(".,")
#     if len(tok) < 2 or re.fullmatch(r"[A-Za-z]\.?", full.split()[0]):
#         return ""
#     return tok


# # =========================================================================== #
# # 1. collect names  (full name for LLM, first name for offline dict)          #
# # =========================================================================== #
# def all_names(df):
#     """Return {full_name: first_name} for every distinct author name."""
#     names = {}
#     for v in df["oa_authors"].dropna():
#         for full in S(v).split("|"):
#             full = full.strip()
#             if full:
#                 names.setdefault(full, first_name(full))
#     print(f"[gender] {len(names):,} unique full author names", file=sys.stderr)
#     return names


# def load_cache():
#     done = {}
#     if Path(NAME_CACHE).exists():
#         for line in open(NAME_CACHE):
#             try:
#                 d = json.loads(line); done[d["name"]] = d
#             except Exception:
#                 pass
#     print(f"[gender] {len(done):,} names cached", file=sys.stderr)
#     return done


# def _append(fh, rec):
#     fh.write(json.dumps(rec, ensure_ascii=False) + "\n"); fh.flush()


# # =========================================================================== #
# # 2. offline dictionary pass (gender-guesser) on the FIRST name               #
# # =========================================================================== #
# def run_offline(names, cache):
#     import gender_guesser.detector as gg
#     det = gg.Detector(case_sensitive=False)
#     conf = {"male": 0.95, "female": 0.95, "mostly_male": 0.75,
#             "mostly_female": 0.75, "andy": 0.5, "unknown": 0.0}
#     todo = [full for full in names if full not in cache]
#     print(f"[offline] {len(todo):,} names to infer", file=sys.stderr)
#     fh = open(NAME_CACHE, "a")
#     for i, full in enumerate(todo, 1):
#         fn = names[full]
#         if not fn:
#             rec = {"name": full, "provider": "none", "gender": "unknown", "prob": 0.0}
#         else:
#             g = det.get_gender(fn)
#             base = "female" if "female" in g else ("male" if "male" in g else "unknown")
#             rec = {"name": full, "provider": "gender_guesser",
#                    "gender": base, "prob": conf.get(g, 0.0)}
#         _append(fh, rec); cache[full] = rec
#         if i % 5000 == 0:
#             print(f"[offline] {i:,}/{len(todo):,}", file=sys.stderr)
#     fh.close()
#     return cache


# # =========================================================================== #
# # 3. dual-LLM consensus pass (OpenAI + DeepSeek TOGETHER) on the residual      #
# # =========================================================================== #
# def _make_client(which):
#     from openai import OpenAI
#     if which == "deepseek":
#         return OpenAI(api_key=_secret("DEEPSEEK_API_KEY"),
#                       base_url="https://api.deepseek.com"), "deepseek-chat"
#     return OpenAI(api_key=_secret("OPENAI_API_KEY")), "gpt-4o-mini"

# SYS = ("You infer likely gender from researcher names, using the WHOLE name "
#        "including family-name origin/language. Return ONLY a JSON array, one object "
#        "per input in the SAME order: "
#        '[{"gender":"male|female|unknown","prob":0.0-1.0}]. '
#        'Use "unknown" for initials-only, unisex, organizational, or names you '
#        "cannot judge. Do NOT guess to fill — reflect low confidence in prob.")

# def _llm_batch(client, model, batch):
#     user = "Names:\n" + "\n".join(f"{i+1}. {n}" for i, n in enumerate(batch))
#     for attempt in range(4):
#         try:
#             r = client.chat.completions.create(
#                 model=model, temperature=0, max_tokens=4000,
#                 messages=[{"role": "system", "content": SYS},
#                           {"role": "user", "content": user}])
#             txt = re.sub(r"^```(json)?|```$", "", r.choices[0].message.content.strip(),
#                          flags=re.MULTILINE).strip()
#             arr = json.loads(txt)
#             if len(arr) == len(batch):
#                 out = []
#                 for o in arr:
#                     g = o.get("gender", "unknown")
#                     out.append((g if g in ("male", "female") else "unknown",
#                                 float(o.get("prob") or 0)))
#                 return out
#         except Exception:
#             time.sleep(2 ** attempt)
#     return [("unknown", 0.0)] * len(batch)

# def run_llm_consensus(cache):
#     """Run BOTH models on names the offline pass left unresolved; keep a gender
#     ONLY where OpenAI and DeepSeek AGREE and both clear PROB_MIN."""
#     oa_client, oa_model = _make_client("openai")
#     ds_client, ds_model = _make_client("deepseek")
#     todo = [n for n, d in cache.items()
#             if d.get("gender") == "unknown" or d.get("prob", 0) < PROB_MIN]
#     print(f"[llm] {len(todo):,} unresolved names -> OpenAI + DeepSeek", file=sys.stderr)
#     fh = open(NAME_CACHE, "a")

#     batches = [todo[i:i + LLM_BATCH] for i in range(0, len(todo), LLM_BATCH)]

#     def process(batch):
#         oa = _llm_batch(oa_client, oa_model, batch)
#         ds = _llm_batch(ds_client, ds_model, batch)
#         recs = []
#         for name, (g1, p1), (g2, p2) in zip(batch, oa, ds):
#             agree = (g1 == g2) and g1 in ("male", "female") and p1 >= PROB_MIN and p2 >= PROB_MIN
#             if agree:
#                 recs.append({"name": name, "provider": "llm_consensus",
#                              "gender": g1, "prob": round(min(p1, p2), 3),
#                              "openai": g1, "deepseek": g2})
#             else:
#                 # record the disagreement but keep it UNKNOWN (do not impute)
#                 recs.append({"name": name, "provider": "llm_nocons",
#                              "gender": "unknown", "prob": 0.0,
#                              "openai": f"{g1}:{p1}", "deepseek": f"{g2}:{p2}"})
#         return recs

#     done = 0
#     with ThreadPoolExecutor(max_workers=LLM_WORKERS) as ex:
#         futs = [ex.submit(process, b) for b in batches]
#         for fut in as_completed(futs):
#             for rec in fut.result():
#                 _append(fh, rec); cache[rec["name"]] = rec
#             done += 1
#             if done % 20 == 0:
#                 print(f"[llm] {done:,}/{len(batches):,} batches", file=sys.stderr)
#     fh.close()
#     return cache


# # =========================================================================== #
# # 4. gated resolution + attach to papers (position-aligned)                    #
# # =========================================================================== #
# def gated(full, cache):
#     d = cache.get(full)
#     if not d or d.get("gender") not in ("male", "female") or d.get("prob", 0) < PROB_MIN:
#         return "unknown"
#     return "M" if d["gender"] == "male" else "F"

# def attach(df, cache):
#     def row(r):
#         authors = [a.strip() for a in S(r.get("oa_authors")).split("|")]
#         pos = S(r.get("au_position_vec")).split("|")
#         first_g = last_g = "unknown"
#         genders = []
#         for i, full in enumerate(authors):
#             if not full:
#                 continue
#             g = gated(full, cache)
#             genders.append(g)
#             p = pos[i] if i < len(pos) else ""
#             if p == "first": first_g = g
#             if p == "last":  last_g = g
#         known = [g for g in genders if g in ("M", "F")]
#         return pd.Series({
#             "first_author_gender": first_g,
#             "last_author_gender": last_g,
#             "share_women_known": (known.count("F") / len(known)) if known else None,
#             "n_authors_gender_known": len(known),
#             "n_authors_total": len([a for a in authors if a]),
#         })
#     out = pd.concat([df, df.apply(row, axis=1)], axis=1)
#     out.to_parquet(OUT_PARQUET, index=False)
#     print(f"[write] {OUT_PARQUET}", file=sys.stderr)

#     resolved = sum(1 for d in cache.values() if d.get("gender") in ("male", "female")
#                    and d.get("prob", 0) >= PROB_MIN)
#     print(f"[coverage] names resolved to M/F: {resolved:,}/{len(cache):,} "
#           f"({resolved/max(len(cache),1):.1%})", file=sys.stderr)
#     return out


# # =========================================================================== #
# # 5. VALIDATION HARNESS — precision + coverage PER BLOC, and go/no-go gate     #
# # =========================================================================== #
# def dominant_bloc(sb):
#     parts = {x for x in S(sb).split(";") if x}
#     for b in ("ssa", "maghreb"):
#         if b in parts: return b
#     return "other"

# def make_validation(out, n_per=80, seed=42):
#     if "study_bloc" not in out.columns:
#         print("[validate] study_bloc absent — run on a parquet with study-location "
#               "to enable per-bloc validation.", file=sys.stderr)
#         return
#     d = out.copy()
#     d["_bloc"] = d["study_bloc"].map(dominant_bloc)
#     parts = [g.sample(min(n_per, len(g)), random_state=seed) for _, g in d.groupby("_bloc")]
#     s = pd.concat(parts).copy()
#     s["first_author_name"] = s["oa_authors"].map(lambda v: S(v).split("|")[0])
#     s = s[["oa_id", "first_author_name", "first_author_gender", "_bloc"]].copy()
#     s["gold_gender"] = ""   # human fills: M / F / U
#     s.to_csv(VALID_CSV, index=False)
#     print(f"[validate] wrote {len(s):,} rows -> {VALID_CSV} (fill gold_gender: M/F/U)",
#           file=sys.stderr)

# def score_validation(path=VALID_CSV):
#     v = pd.read_csv(path, dtype=str).fillna("")
#     print("\n[validation] per bloc (precision among M/F guesses; coverage = share guessed):")
#     print(f"{'bloc':<9}{'n':>6}{'coverage':>10}{'precision':>11}")
#     gate = {}
#     for bloc, g in v.groupby("_bloc"):
#         guessed = g[g["first_author_gender"].isin(["M", "F"])]
#         coverage = len(guessed) / len(g) if len(g) else 0
#         gold_ok = guessed[guessed["gold_gender"].isin(["M", "F"])]
#         correct = (gold_ok["first_author_gender"] == gold_ok["gold_gender"]).sum()
#         precision = correct / len(gold_ok) if len(gold_ok) else float("nan")
#         gate[bloc] = coverage
#         print(f"{bloc:<9}{len(g):>6}{coverage:>10.1%}{precision:>11.3f}")
#     if "ssa" in gate and "other" in gate:
#         gap = abs(gate.get("ssa", 0) - gate.get("other", 0))
#         verdict = "OK to compare blocs" if gap < 0.15 else \
#                   "DO NOT compare blocs — coverage gap too large; report global/North only"
#         print(f"\n[gate] SSA vs other coverage gap = {gap:.1%}  ->  {verdict}")


# # =========================================================================== #
# if __name__ == "__main__":
#     df_all = pd.read_parquet(IN_PARQUET)
#     df = df_all[df_all["llm_label"].notna()]
#     print(f"[in] {len(df):,} rows", file=sys.stderr)
#     names = all_names(df)
#     cache = load_cache()
#     if RUN_OFFLINE:
#         cache = run_offline(names, cache)
#     if RUN_LLM:
#         cache = run_llm_consensus(cache)
#     out = attach(df, cache)
#     make_validation(out)
#     # after a human fills gold_gender in VALID_CSV:  score_validation()